In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset
from utils.hub_card import push_dataset_card
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# ED vital signs time series from ed.vitalsign.
# One row per vitals measurement per ED visit, ordered by charttime.
# Multiple readings per visit — tracks patient deterioration/improvement over the ED stay.
# rhythm is cardiac rhythm (e.g. 'Sinus Rhythm', 'Atrial Fibrillation') — only in vitalsign, not triage.
# Used as time-varying state features in the RL model.

query_vitals = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT
  v.stay_id   AS ed_stay_id,
  v.subject_id,
  v.charttime,
  v.temperature,
  v.heartrate,
  v.resprate,
  v.o2sat,
  v.sbp,
  v.dbp,
  v.rhythm,
  v.pain
FROM `physionet-data.mimiciv_ed.vitalsign` v
INNER JOIN cohort_subjects cs ON v.stay_id = cs.ed_stay_id
ORDER BY v.stay_id, v.charttime
"""

print("Running vitals query...")
df_vitals = client.query(query_vitals).to_dataframe()
print(f"Shape: {df_vitals.shape}")
print(f"Unique ED visits with vitals: {df_vitals['ed_stay_id'].nunique():,}")
print(f"Avg vital readings per visit: {len(df_vitals) / df_vitals['ed_stay_id'].nunique():.1f}")
df_vitals.head()

In [ ]:
import pickle

# Remap stay_id for consecutive ED visits collapsed in the cohort pipeline.
# Any vitals row whose ed_stay_id was a "second" visit gets remapped to the first.
with open("stay_id_remap.pkl", "rb") as f:
    stay_id_remap = pickle.load(f)

df_vitals['ed_stay_id'] = df_vitals['ed_stay_id'].replace(stay_id_remap)
print(f"Remapped {sum(df_vitals['ed_stay_id'].isin(stay_id_remap)):,} rows (expect 0 remaining unmapped)")

In [ ]:
DESCRIPTION = (
    "ED vital signs time series from ed.vitalsign, joined to the cohort by ed_stay_id. "
    "One row per vitals measurement per ED visit. stay_id remapping applied so that "
    "vitals from a patient's second consecutive ED visit are attributed to the first (merged) stay_id."
)

ds = Dataset.from_pandas(df_vitals, split='vitals')
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="vitals", data_dir="vitals")
push_dataset_card("ADS599-Capstone/raw_data", config_name="vitals", split="vitals", data_dir="vitals", description=DESCRIPTION)
print("Pushed vitals to HuggingFace Hub.")